# Week 1 — Foundation
Validate and explore the fraud dataset, reserve a global test set, and generate six non-IID client training partitions.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import load_and_validate, save_week1_outputs, summarize_dataset

DATA = ROOT / 'data/raw/creditcard.csv'
OUT = ROOT / 'data/processed'
FIGURES = ROOT / 'reports/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

## Validate and summarize
The loader checks the exact schema, numeric values, missing data, and binary target before analysis.

In [ ]:
df = load_and_validate(DATA)
summary = summarize_dataset(df)
display(pd.Series(summary, name='value').to_frame())
counts = df['Class'].value_counts().sort_index().rename(index={0: 'Normal', 1: 'Fraud'})
display(counts.to_frame('count').assign(percent=lambda x: 100*x['count']/len(df)))
ax = sns.barplot(x=counts.index, y=counts.values)
ax.set(title='Class imbalance (log scale)', ylabel='Transactions', yscale='log')
plt.tight_layout(); plt.savefig(FIGURES / 'class_imbalance.png', dpi=160); plt.show()

## Feature distributions and target correlations
These are descriptive checks. Any model preprocessing must later be fitted on training data only.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x='Amount', hue='Class', bins=80, log_scale=(True, False), ax=axes[0])
sns.histplot(data=df, x='Time', hue='Class', bins=80, stat='density', common_norm=False, ax=axes[1])
axes[0].set_title('Amount by class (log x)'); axes[1].set_title('Time by class')
plt.tight_layout(); plt.savefig(FIGURES / 'amount_time_distributions.png', dpi=160); plt.show()
corr = df.corr(numeric_only=True)['Class'].drop('Class').sort_values(key=abs)
top = pd.concat([corr.head(8), corr.tail(8)]).drop_duplicates().sort_values()
plt.figure(figsize=(8, 6)); sns.barplot(x=top.values, y=top.index, orient='h')
plt.title('Features most correlated with Class'); plt.xlabel('Pearson correlation')
plt.tight_layout(); plt.savefig(FIGURES / 'target_correlations.png', dpi=160); plt.show()
display(top.to_frame('correlation'))

## Leakage-safe non-IID partitioning
Reserve a stratified 20% test set first. Partition only training rows with a class-wise Dirichlet distribution (alpha 0.5), requiring at least 10 fraud cases per client.

In [ ]:
manifest = save_week1_outputs(df, OUT, num_clients=6, alpha=0.5, test_size=0.2, seed=42, min_fraud_per_client=10)
display(manifest.style.format({'fraud_rate': '{:.4%}'}))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=manifest, x='client_id', y='rows', ax=axes[0]); axes[0].set_title('Rows per client')
sns.barplot(data=manifest, x='client_id', y='fraud_rate', ax=axes[1])
axes[1].axhline(df['Class'].mean(), color='red', linestyle='--', label='Global rate')
axes[1].set_title('Fraud rate per client'); axes[1].legend()
plt.tight_layout(); plt.savefig(FIGURES / 'client_heterogeneity.png', dpi=160); plt.show()

## Completion check
Review the saved manifest and figures, confirm every client has fraud examples and visibly different prevalence, then save this notebook with outputs. Keep `global_test.csv` unchanged for every later model comparison.